### Uso de TensorFlow para Redes Neuronales

## Objetivo
Tomando los siguiente parámetros:

- Entrada: 3 valores → a, b, resultado
- Salida: 1 entre 4 clases → 0 = suma, 1 = resta, 2 = multiplicación, 3 = división

El script genera los datos, entrena la red, evalúa y permite hacer predicciones interactivas.

## Instalamos TensorFlow

In [2]:
# Instalamos TensorFlow y dependencias
# Posteriormente tendrás que reiniciar el entorno de ejecución
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.20.0-cp313-cp313-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.2.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.76.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.12.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached ml_dtypes-0.5.3-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cac

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Ejecuta el script: entrenará en pocos segundos y mostrará ejemplos.

Llama a *clasificar_operacion(a, b, resultado)* desde tu propio código o consola.

La red aprenderá a distinguir las cuatro operaciones con una precisión cercana al 100 % en datos de test.

In [44]:
"""
Red neuronal que clasifica la operación básica (suma, resta, multiplicación, división)
realizada entre dos números (a, b) a partir del resultado.

Entrada : vector de 3 valores → [a, b, resultado]
Salida  : probabilidad sobre 4 clases → 0=suma, 1=resta, 2=multiplicación, 3=división
"""

# ------------------------------------------------------------------
# 1.  IMPORTS
# ------------------------------------------------------------------
import numpy as np                 # Crear y manipular arrays
import tensorflow as tf            # Motor de deep-learning
from tensorflow.keras import layers, models   # API de alto nivel que facilita creación de redes

# Fijamos semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# ------------------------------------------------------------------
# 2.  GENERACIÓN DE DATOS SINTÉTICOS
# ------------------------------------------------------------------
# ¿Por qué sintéticos?
#   - No existe un data-set público con millones de tripletas (a,b,res) etiquetadas.
#   - Podemos generar tantos como queramos sin coste.
#
# ¿Por qué 100 000 muestras?
#   - Es suficiente para que la red vea muchas combinaciones de operandos y
#     resultados, evitando overfitting a pocos casos.
#   - En CPU tarda <1 minuto entrenar.
def generar_datos(n=100_000):
    """
    Devuelve:
        X : ndarray shape (n, 3) → columnas [a, b, resultado]
        y : ndarray shape (n,)   → clase 0..3
    """
    # 2.1  Operandos
    # Rango [-50, 50] para que la red vea números positivos, negativos,
    # grandes y pequeños.  Un rango más estrecho haría que la red
    # memorizara patrones locales; uno más amplio puede introducir
    # desbordamientos (inf) en división.
    a = np.random.uniform(-50, 50, size=n)
    b = np.random.uniform(-50, 50, size=n)

    # 2.2  Corrección de división por cero
    # Si |b| ≈ 0, forzamos un valor pequeño para evitar NaN/inf.
    mask = np.abs(b) < 1e-4
    b[mask] = 1e-4          # 1e-4 es pequeño pero no desaparece en float32

    # 2.3  Calculamos las cuatro operaciones para todas las muestras
    ops = {
        0: a + b,
        1: a - b,
        2: a * b,
        3: a / b
    }

    # 2.4  Elegimos UNA operación por muestra → target y
    # Usamos uniforme para que las 4 clases estén balanceadas (~25 % cada una).
    y = np.random.randint(0, 4, size=n)

    # 2.5  Extraemos el resultado que corresponde a la operación elegida
    resultado = np.array([ops[y[i]][i] for i in range(n)])

    # 2.6  Componemos la matriz de entrada
    X = np.stack([a, b, resultado], axis=-1).astype(np.float32)
    return X, y

# 2.7  ¿Por qué float32 y no float64?
#   - Reducimos a la mitad el consumo de memoria.
#   - TensorFlow utiliza float32 por defecto en GPU.
X_train, y_train = generar_datos(2_000_000)
X_test,  y_test  = generar_datos(200_000)

# ------------------------------------------------------------------
# 3.  NORMALIZACIÓN (OPCIONAL PERO RECOMENDADA)
# ------------------------------------------------------------------
# En problemas reales conviene escalar las entradas al rango [-1,1] o [0,1].
# Aquí los rangos de a, b y resultado son similares, por lo que la red
# converge igual; pero si ampliáramos rangos convendría añadir:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler().fit(X_train)
# X_train = scaler.transform(X_train)

# ------------------------------------------------------------------
# 4.  ARQUITECTURA DE LA RED
# ------------------------------------------------------------------
# 4.1  ¿Por qué 3 neuronas de entrada?
#   - Una por cada feature: a, b, resultado.
# 4.2  ¿Por qué 4 neuronas de salida?
#   - Una por clase → softmax devuelve probabilidad de cada clase.
# 4.3  ¿Por qué dos capas ocultas (128 y 64)?
#   - Capacidad suficiente para aprender fronteras no lineales entre
#     las 4 operaciones sin llegar a redes enormes que overfitten.
#   - Se puede probar otras combinaciones (256,128) o añadir capas,
#     pero estas cifras dan >99 % de accuracy en pocos epochs.
# 4.4  ¿Por qué ReLU?
#   - Simple, rápida de calcular, no sufre saturación tan rápido como sigmoide/tanh.
#   - sigmoide, se usaría en la capa de salida para problemas binarios.
#   - tanh, es similar a ReLU pero más lenta y con saturación.
model = models.Sequential([
    layers.Input(shape=(3,)),          # entrada explícita
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(4,   activation='softmax')  # probabilidades
])

# ------------------------------------------------------------------
# 5.  FUNCIÓN DE PÉRDIDA Y OPTIMIZADOR
# ------------------------------------------------------------------
# 5.1  ¿sparse_categorical_crossentropy?
#   - Las etiquetas y_train son enteros 0-3 (no one-hot).
#   - Esta versión es más eficiente que categorical_crossentropy
#     porque evita convertir a one-hot internamente.
# 5.2  ¿Adam?
#   - Adapta la tasa de aprendizaje por coordenada → converge rápido
#     sin necesidad de ajustar learning-rate a mano.
#   - Otras opciones: SGD (más lento), RMSprop (similar a Adam).
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ------------------------------------------------------------------
# 6.  ENTRENAMIENTO
# ------------------------------------------------------------------
# 6.1  epochs = 5
#   - Con 200 k muestras basta; después de 5 épocas la pérdida ya es <0.01.
# 6.2  batch_size = 512
#   - Tamaño grande acelera el paso por epoch y aprovecha vectorización.
#   - En CPU suele ser óptimo 256-1024; en GPU puede ser mayor.
# 6.3  validation_split = 0.1
#   - Reservamos 10 % de los datos de entreno para validar que no
#     sobreajustamos en cada epoch.
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=1024,
                    validation_split=0.1,
                    verbose=2)          # 2 → una línea por epoch

# ------------------------------------------------------------------
# 7.  EVALUACIÓN
# ------------------------------------------------------------------
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nPrecisión en test: {test_acc:.4f}")
# Guardamos el modelo entrenado para uso posterior en formato keras
model.save("modelo_operaciones.keras")
print("Modelo guardado en 'modelo_operaciones.keras'")

# ------------------------------------------------------------------
# 8.  FUNCIÓN DE PREDICCIÓN PARA EL USUARIO
# ------------------------------------------------------------------
def clasificar_operacion(a, b, res):
    """
    Devuelve la operación más probable dado (a, b, resultado).
    """
    pred = model.predict(np.array([[a, b, res]], dtype=np.float32))
    clases = ['suma', 'resta', 'multiplicación', 'división']
    return clases[np.argmax(pred)]

# ------------------------------------------------------------------
# 6.  TRADUCCIÓN ÍNDICE → TEXTO
# ------------------------------------------------------------------
# Diccionario de traducción de la clase predicha (índice) a cadena legible
IDX_A_OPERACION = {
    0: "suma",
    1: "resta",
    2: "multiplicación",
    3: "división"
}

# ------------------------------------------------------------------
# 7.  FUNCIÓN DE PREDICCIÓN CON TEXTO
# ------------------------------------------------------------------
def clasificar_operacion(a, b, res):
    """
    Devuelve la operación más probable en formato textual.
    Ejemplo:  clasificar_operacion(6, 2, 12) -> 'multiplicación'
    """
    pred = model.predict(np.array([[a, b, res]], dtype=np.float32))
    print(pred)
    clase_idx = int(np.argmax(pred))          # convertimos a int por claridad
    return IDX_A_OPERACION[clase_idx]


Epoch 1/10
1758/1758 - 3s - 2ms/step - accuracy: 0.9499 - loss: 0.2631 - val_accuracy: 0.9736 - val_loss: 0.1809
Epoch 2/10
1758/1758 - 3s - 1ms/step - accuracy: 0.9777 - loss: 0.0967 - val_accuracy: 0.9781 - val_loss: 0.1280
Epoch 3/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9822 - loss: 0.0518 - val_accuracy: 0.9826 - val_loss: 0.0419
Epoch 4/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9834 - loss: 0.0836 - val_accuracy: 0.9820 - val_loss: 0.1292
Epoch 5/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9856 - loss: 0.0477 - val_accuracy: 0.9863 - val_loss: 0.0315
Epoch 6/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9881 - loss: 0.0290 - val_accuracy: 0.9896 - val_loss: 0.0251
Epoch 7/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9893 - loss: 0.0438 - val_accuracy: 0.9904 - val_loss: 0.0220
Epoch 8/10
1758/1758 - 2s - 1ms/step - accuracy: 0.9912 - loss: 0.0227 - val_accuracy: 0.9920 - val_loss: 0.0192
Epoch 9/10
1758/1758 - 3s - 2ms/step - accuracy: 0.9915 - loss: 0.0275 - val_accuracy: 0.9923 - 

In [ ]:
# ------------------------------------------------------------------
# 9.  DEMO RÁPIDA
# ------------------------------------------------------------------

# Cargamos el modelo entrenado (si se ejecuta en otro entorno)
# from tensorflow.keras.models import load_model
# model = load_model("modelo_operaciones.keras")

# Simulamos que esta celda es un script independiente

if __name__ == "__main__":
    print("\nEjemplos de clasificación:")
    for _ in range(5):
        a, b = np.round(np.random.uniform(-20, 20, 2), 2)
        res = a + b
        print(f"a={a}, b={b}, res={res} → {clasificar_operacion(a, b, res)}")

In [42]:
a = 100
b = 1
res = 101
print(f"a={a}, b={b}, res={res} → {clasificar_operacion(a, b, res)}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[9.9889863e-01 1.0956675e-03 5.0467156e-06 6.4139090e-07]]
a=100, b=1, res=101 → suma
